# SL-7 Cross-Model Run — SmolVLM2-2.2B-Instruct (family-gated protocol)

Model: `HuggingFaceTB/SmolVLM2-2.2B-Instruct`

This notebook uses the **exact frozen SL-7 v16 images, prompts, answer keys, family confirmation thresholds, greedy decoding, and max_new_tokens=64**.

### Cross-model calibration rule
Calibration is evaluated **per prerequisite skill**. A failed calibration skill makes only the blind-spot families that depend on that skill **INELIGIBLE / diagnostic only**; unrelated families remain eligible. The notebook therefore does not stop merely because one unrelated calibration family fails.

No model-specific image, prompt, threshold, answer-key, or scored-item changes are made.


> **Kaggle dependency fix:** this version installs `num2words` before `AutoProcessor.from_pretrained(...)`, which is required by the current SmolVLM processor implementation.


> **Scorer patch:** this copy is based on your executed notebook. Only the final scorer cell is changed to safely handle families with zero eligible items.


In [1]:
PROTOCOL_VERSION = "SL7_CROSS_MODEL_FAMILY_GATED_V2"
print("="*76)
print(PROTOCOL_VERSION)
print("Model: SmolVLM2-2.2B-Instruct")
print("Calibration failures gate only dependent SL-7 families; the holdout still runs.")
print("="*76)


SL7_CROSS_MODEL_FAMILY_GATED_V2
Model: SmolVLM2-2.2B-Instruct
Calibration failures gate only dependent SL-7 families; the holdout still runs.


In [2]:
# SmolVLM processor dependency required by current Transformers builds.
%pip install -q num2words

import num2words
print("num2words installed successfully")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 2.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
num2words installed successfully


In [3]:
!pip install -q -U "transformers==5.17.0" "accelerate>=1.0" "huggingface_hub>=0.30"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 27.6 MB/s eta 0:00:00


In [4]:
import os, glob, csv, json, time, platform, sys, zipfile, importlib, random, gc
from pathlib import Path
from PIL import Image
import torch, transformers

MAX_NEW_TOKENS=64
INFERENCE_SEED=20261004
random.seed(INFERENCE_SEED); torch.manual_seed(INFERENCE_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(INFERENCE_SEED)

hits=[]
for p in glob.glob("/kaggle/input/**/dataset/items.csv", recursive=True):
    root=Path(p).parent.parent
    if (root/"VERSION_SL7_v16.txt").exists() and (root/"eval/score.py").exists():
        hits.append(root)
if not hits:
    raise FileNotFoundError("Add the snakes-ladders-SL7-multimodel-v1 package as a Kaggle input dataset.")
ROOT=hits[0]
ITEMS=list(csv.DictReader(open(ROOT/"dataset/items.csv",encoding="utf-8")))
CAL=[x for x in ITEMS if x["part"]=="CAL"]
SCORED=[x for x in ITEMS if x["part"]=="SCORED"]
print("Frozen project root:",ROOT)
print("Calibration:",len(CAL),"Scored:",len(SCORED))

sys.path.insert(0,str(ROOT/"eval"))
import score
importlib.reload(score)


Frozen project root: /kaggle/input/datasets/sabahnushra/snake-ladder-multimodel-dataset/snakes-ladders-SL7-multimodel-v1
Calibration: 20 Scored: 150


<module 'score' from '/kaggle/input/datasets/sabahnushra/snake-ladder-multimodel-dataset/snakes-ladders-SL7-multimodel-v1/eval/score.py'>

In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText
MODEL_ID='HuggingFaceTB/SmolVLM2-2.2B-Instruct'
MODEL_SLUG='smolvlm2_2_2b'
MODEL_DISPLAY='SmolVLM2-2.2B-Instruct'
OUT=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}")
OUT.mkdir(exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU accelerator.")

# Optional HF login for gated checkpoints (especially Gemma).
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

load_kwargs=dict(device_map={"":0}, dtype=torch.float16, low_cpu_mem_usage=True)
try:
    model=AutoModelForImageTextToText.from_pretrained(MODEL_ID,attn_implementation="sdpa",trust_remote_code=False,**load_kwargs)
    attention="sdpa"
except Exception as e:
    print("SDPA load failed; retrying with model default attention:",repr(e))
    model=AutoModelForImageTextToText.from_pretrained(MODEL_ID,trust_remote_code=False,**load_kwargs)
    attention="model_default"
model.eval()
processor=AutoProcessor.from_pretrained(MODEL_ID,trust_remote_code=False)

env={"model_key":'smolvlm2_2_2b',"model_display":MODEL_DISPLAY,"model_id":MODEL_ID,"parameters":sum(p.numel() for p in model.parameters()),"transformers":transformers.__version__,"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0),"python":platform.python_version(),"dtype":"float16","attention":attention,"decoding":"greedy","max_new_tokens":MAX_NEW_TOKENS,"inference_seed":INFERENCE_SEED,"dataset":"SL7_v16_frozen"}
(OUT/"environment.json").write_text(json.dumps(env,indent=2)); print(json.dumps(env,indent=2))


config.json:   0%|          | 0.00/3.64k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


model.safetensors.index.json:   0%|          | 0.00/63.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/28.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

{
  "model_key": "smolvlm2_2_2b",
  "model_display": "SmolVLM2-2.2B-Instruct",
  "model_id": "HuggingFaceTB/SmolVLM2-2.2B-Instruct",
  "parameters": 2246784880,
  "transformers": "5.17.0",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "python": "3.13.15",
  "dtype": "float16",
  "attention": "sdpa",
  "decoding": "greedy",
  "max_new_tokens": 64,
  "inference_seed": 20261004,
  "dataset": "SL7_v16_frozen"
}


In [6]:
def run_item(it):
    content=[]
    for rel in [p for p in it["image_paths"].split("|") if p]:
        p=str(ROOT/rel)
        if not Path(p).exists(): raise FileNotFoundError(p)
        content.append({"type":"image","path":p})
    content.append({"type":"text","text":it["prompt"]})
    messages=[{"role":"user","content":content}]
    t0=time.time()
    inputs=processor.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt")
    try:
        inputs=inputs.to(model.device,dtype=torch.float16)
    except TypeError:
        inputs=inputs.to(model.device)
        for k,v in list(inputs.items()):
            if torch.is_tensor(v) and torch.is_floating_point(v): inputs[k]=v.to(torch.float16)
    inputs.pop("token_type_ids",None)
    input_len=inputs["input_ids"].shape[1]
    try:
        with torch.inference_mode():
            generated=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False)
        new=generated[:,input_len:]
        reply=processor.batch_decode(new,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()
        return {"item_id":it["item_id"],"raw_output":reply,"seconds":round(time.time()-t0,2)}
    finally:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [7]:
RAW=OUT/"raw_outputs.csv"
outrows=[]

def save_raw():
    with open(RAW,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["item_id","raw_output","seconds"])
        w.writeheader(); w.writerows(outrows)

# Frozen calibration is always run first.
for idx,it in enumerate(CAL,1):
    outrows.append(run_item(it)); save_raw()
    print(f"CAL {idx}/{len(CAL)}",it["item_id"],"=>",outrows[-1]["raw_output"][:160])

pred={r["item_id"]:r["raw_output"] for r in outrows}
groups={}
for it in CAL:
    parsed=score.parse_strict(pred[it["item_id"]],it["answer_format"])
    ok=parsed.lower()==it["correct_answer"].lower() if parsed else False
    groups.setdefault(it["task_family"],[]).append(ok)
cal_counts={k:(sum(v),len(v)) for k,v in groups.items()}
requirements={"CAL_number":7,"CAL_endpoint":3,"CAL_die":4,"CAL_multi_image":3}
cal_pass={k:(sum(groups.get(k,[]))>=need) for k,need in requirements.items()}
failed=[k for k,v in cal_pass.items() if not v]

# Same family-specific prerequisite map used for the cross-model study.
family_gate_requirements={
    "B1_zigzag_direction":["CAL_number"],
    "B2_hidden_reconstruction":["CAL_number"],
    "B3_endpoint_anchoring":["CAL_endpoint"],
    "B4_perception_to_action":["CAL_number","CAL_endpoint"],
    "B5_multi_image_position":["CAL_number","CAL_endpoint","CAL_multi_image"],
    "B6_conditioned_counting":["CAL_number","CAL_endpoint"],
    "B7_multi_source_composition":["CAL_number","CAL_die","CAL_multi_image"],
}
family_eligibility={fam:all(cal_pass[g] for g in gates) for fam,gates in family_gate_requirements.items()}

cal_record={
    "counts":{k:list(v) for k,v in cal_counts.items()},
    "requirements":requirements,
    "calibration_pass":cal_pass,
    "original_global_gate_passed":not failed,
    "failed_calibrations":failed,
    "cross_model_protocol":"family_specific_calibration_gating_v2",
    "family_gate_requirements":family_gate_requirements,
    "family_eligibility":family_eligibility,
}
(OUT/"calibration_status.json").write_text(json.dumps(cal_record,indent=2))

print("\nCalibration:",cal_counts)
print("Calibration pass map:",cal_pass)
print("\nFamily eligibility:")
for fam,ok in family_eligibility.items():
    print(" ",fam,":","ELIGIBLE" if ok else "INELIGIBLE / diagnostic only",family_gate_requirements[fam])
if failed:
    print("\nOriginal global gate would fail for:",failed)
    print("FAMILY-GATED MODE: continuing to the frozen 150-item holdout.")
else:
    print("\nAll calibration gates passed; all seven families are eligible.")


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


CAL 1/20 CAL_NUM_01 => 8


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


CAL 2/20 CAL_NUM_02 => 15


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

CAL 17/20 CAL_CMP_01 => B


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


CAL 18/20 CAL_CMP_02 => B


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


CAL 19/20 CAL_CMP_03 => B
CAL 20/20 CAL_CMP_04 => B

Calibration: {'CAL_number': (2, 8), 'CAL_endpoint': (2, 4), 'CAL_die': (3, 4), 'CAL_multi_image': (2, 4)}
Calibration pass map: {'CAL_number': False, 'CAL_endpoint': False, 'CAL_die': False, 'CAL_multi_image': False}

Family eligibility:
  B1_zigzag_direction : INELIGIBLE / diagnostic only ['CAL_number']
  B2_hidden_reconstruction : INELIGIBLE / diagnostic only ['CAL_number']
  B3_endpoint_anchoring : INELIGIBLE / diagnostic only ['CAL_endpoint']
  B4_perception_to_action : INELIGIBLE / diagnostic only ['CAL_number', 'CAL_endpoint']
  B5_multi_image_position : INELIGIBLE / diagnostic only ['CAL_number', 'CAL_endpoint', 'CAL_multi_image']
  B6_conditioned_counting : INELIGIBLE / diagnostic only ['CAL_number', 'CAL_endpoint']
  B7_multi_source_composition : INELIGIBLE / diagnostic only ['CAL_number', 'CAL_die', 'CAL_multi_image']

Original global gate would fail for: ['CAL_number', 'CAL_endpoint', 'CAL_die', 'CAL_multi_image']
FAMILY-G

In [8]:
for idx,it in enumerate(SCORED,1):
    outrows.append(run_item(it)); save_raw()
    if idx%10==0 or idx==len(SCORED): print(f"SCORED {idx}/{len(SCORED)} complete")
print("Raw outputs saved:",RAW)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

SCORED 10/150 complete


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

SCORED 20/150 complete


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

...

SCORED 130/150 complete


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

SCORED 140/150 complete


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


SCORED 150/150 complete
Raw outputs saved: /kaggle/working/results_SL7_smolvlm2_2_2b/raw_outputs.csv


In [ ]:
# ---- scorer-only robustness patch ----
# Some models can have 0 eligible items for one or more families.
# The frozen scorer returns empty-string CI bounds for n=0, then its Markdown
# formatter tries to print them with ":.1f", causing a ValueError.
# This patch changes ONLY that presentation edge case; it does not change
# predictions, parsing, accuracy, eligibility, thresholds, or confirmation.

_original_wilson = score.wilson

def _wilson_zero_safe(k, n, *args, **kwargs):
    if n == 0:
        return (float("nan"), float("nan"))
    return _original_wilson(k, n, *args, **kwargs)

score.wilson = _wilson_zero_safe

scored_rows,family_rows,original_cal_pass=score.score(ROOT/"dataset/items.csv",RAW,OUT)

# Replace nan CI text in the human-readable summary only.
_summary_path = OUT/"summary.md"
if _summary_path.exists():
    _txt = _summary_path.read_text(encoding="utf-8")
    _txt = _txt.replace("nan–nan%", "N/A").replace("nan-nan%", "N/A")
    _summary_path.write_text(_txt, encoding="utf-8")

# Apply only the cross-model calibration eligibility overlay; the frozen family
# scoring/confirmation rules remain unchanged.
fam_csv=OUT/"family_summary.csv"
fam_rows=list(csv.DictReader(open(fam_csv,encoding="utf-8")))
for r in fam_rows:
    fam=r["family"]
    eligible=bool(family_eligibility.get(fam,False))
    scorer_confirmed=(r["confirmed"]=="1")
    r["calibration_eligible"]="1" if eligible else "0"
    r["headline_confirmed"]="1" if (eligible and scorer_confirmed) else "0"
    reqs=family_gate_requirements.get(fam,[])
    failed_reqs=[g for g in reqs if not cal_pass.get(g,False)]
    r["eligibility_note"]="eligible" if eligible else "failed calibration: "+",".join(failed_reqs)

fg_csv=OUT/"family_summary_family_gated.csv"
fields=list(fam_rows[0].keys())
with open(fg_csv,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=fields)
    w.writeheader()
    w.writerows(fam_rows)

L=[
    "# SL-7 Cross-Model Family-Gated Summary","",
    f"Model: **{MODEL_DISPLAY}**","",
    "Frozen SL-7 v16 stimuli and blind-spot confirmation thresholds are unchanged. "
    "Calibration eligibility is applied per dependent family.","",
    "## Calibration","",
    "| Calibration | Correct | Required | Pass |",
    "|---|---:|---:|---|"
]
for name in ["CAL_number","CAL_endpoint","CAL_die","CAL_multi_image"]:
    k,n=cal_counts.get(name,(0,0))
    need=requirements[name]
    L.append(
        f"| {name.replace('CAL_','')} | {k}/{n} | {need}/{n} | "
        f"{'PASS' if cal_pass[name] else 'FAIL'} |"
    )

L += [
    "",
    "## Family results","",
    "| Family | Eligible? | Frozen confirmed? | Headline confirmed? | "
    "Control/component | Probe | Gap | Note |",
    "|---|---|---|---|---:|---:|---:|---|"
]
for r in fam_rows:
    elig="YES" if r["calibration_eligible"]=="1" else "NO"
    conf="YES" if r["confirmed"]=="1" else "NO"
    head="YES" if r["headline_confirmed"]=="1" else "NO"
    L.append(
        f"| {r['family']} | {elig} | {conf} | **{head}** | "
        f"{r['control_or_component_correct']}/{r['control_or_component_n']} | "
        f"{r['probe_correct']}/{r['probe_n']} | {r['gap_percentage_points']} pp | "
        f"{r['eligibility_note']} |"
    )

L += [
    "",
    "`headline_confirmed=1` requires both the unchanged frozen family criterion "
    "and all directly relevant calibration prerequisites.",
    ""
]

summary_fg=OUT/"summary_family_gated.md"
summary_fg.write_text("\n".join(L),encoding="utf-8")
print(summary_fg.read_text())

zip_path=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}_family_gated.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=p.name)
print("Download:",zip_path)
